# TP3 : consumer groups, offsets, rétention et compaction

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 14:05-15:00, après la démo du TP (14:00-14:05) |
| Compétences | C5 Produire, consommer, gérer les groupes et les offsets ; C6 Rétention et compaction |
| Slides de rappel | S11 (consumer groups), S12 (rééquilibrage et KIP-848), S29 (commits et `auto.offset.reset`), S31 (rétention et compaction) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule, avec trois terminaux |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez les trois cellules du début (connexion, fonctions d'aide, outils du TP), puis les cellules « Code fourni » des étapes déjà faites. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, envoi de messages) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une cellule **`# TODO`**, puis une **vérification**.
- **Vous n'avez pas de code à écrire.** Dans une cellule `# TODO`, remplacez chaque `...` par une valeur : un **réglage Kafka** (`"earliest"`, `"compact"`...) ou votre **prévision** du résultat (un nombre, `True` ou `False`). Les noms qui commencent par `PREDICTED_` sont des prévisions : décidez-les avant d'exécuter, la vérification les compare au résultat observé.
- Sous les réponses, le code est fourni : lisez ses commentaires, ne le modifiez pas. Tant qu'il reste un `...`, la ligne `if filled(...):` l'empêche de s'exécuter et affiche « À compléter ».
- Les commandes en ligne de commande qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- La version Markdown de ce TP utilise trois terminaux pour faire tourner plusieurs consommateurs. Ici, chaque consommateur est une **instance** qui tourne en arrière-plan dans le notebook (classe `Instance`, étape 6.3).

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Trois fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

def filled(*answers):
    """Vrai si toutes les réponses d'une cellule TODO sont données ; sinon affiche « À compléter »."""
    for answer in answers:
        if answer is ...:
            print("À compléter : remplacez chaque ... de la cellule par votre réponse, puis exécutez-la de nouveau.")
            return False
    return True

In [ ]:
# Outils du TP : imports, lecture des offsets (fonctions du TP2), producteur partagé
import csv
import re
import threading
import time

from confluent_kafka import (OFFSET_BEGINNING, Consumer, ConsumerGroupState, ConsumerGroupTopicPartitions,
                             ConsumerGroupType, Producer, TopicPartition)
from confluent_kafka.admin import ConfigResource, NewTopic, OffsetSpec


def topic_offsets(topic, spec):
    """{partition: offset} pour une spécification : OffsetSpec.earliest(), latest() ou for_timestamp(ms)."""
    admin = AdminClient(CONF)
    # Nombre de partitions du topic, lu dans les métadonnées du cluster
    num_partitions = len(admin.list_topics(topic, timeout=10).topics[topic].partitions)
    # Une seule requête pour toutes les partitions ; la réponse contient un futur par partition
    futures = admin.list_offsets({TopicPartition(topic, p): spec for p in range(num_partitions)})
    # f.result() attend la réponse du broker ; les offsets sont rangés par numéro de partition
    return {tp.partition: f.result().offset for tp, f in sorted(futures.items(), key=lambda x: x[0].partition)}


def end_offsets(topic):
    """{partition: offset de fin} : l'offset que recevra le prochain message de chaque partition."""
    return topic_offsets(topic, OffsetSpec.latest())


# Producteur partagé par tout le TP. murmur2_random range les clés comme les outils Java (TP2) :
# une même clé va dans la même partition, en Python comme en ligne de commande.
producer = Producer({"bootstrap.servers": BS, "partitioner": "murmur2_random", "log_level": 4})


def send(topic, key, values):
    """Envoie chaque valeur de la liste avec la même clé, puis attend la confirmation du cluster."""
    for value in values:
        producer.produce(topic, key=key, value=value)
    producer.flush(30)  # attend que tous les messages soient confirmés, 30 s au plus


print("Bootstrap INTERNAL :", BS)
print("orders :", end_offsets("orders"))

## 1. Objectifs

À la fin du TP, vous savez :

- consommer avec un consumer group et calculer son lag ;
- observer la répartition des partitions entre consommateurs et le rééquilibrage ;
- reconnaître un groupe utilisant le nouveau protocole (KIP-848) ;
- repositionner les offsets d'un groupe en sécurité : simulation, application, export ;
- faire supprimer des données par la rétention en temps ;
- configurer un topic compacté et supprimer une clé avec un tombstone.

## 2. Prérequis

- TP2 terminé : topic `orders` avec 6 partitions et 60 messages (voir la cellule précédente). Sinon, demandez au formateur le bloc de rattrapage du TP2.
- Jeu de données `lab/data/product-prices.txt` : 20 prix pour 4 produits `P-100` à `P-400`.
- Notions : slides S11, S29, S31.

## 3. Architecture utilisée

```
                       orders (6 partitions)
          p0    p1    p2    p3    p4    p5
           \     |     /     \     |     /
   groupe analytics : instance a (p0, p1, p2) + instance b (p3, p4, p5)
   groupe billing   : un consommateur, lancé puis arrêté (offsets conservés)

   Offsets validés : topic interne __consumer_offsets (50 partitions, compacté)
   Coordinateur du groupe : le leader de la partition de __consumer_offsets
   qui correspond au nom du groupe.
```

| Topic | Partitions | Configuration | Usage |
|---|---|---|---|
| `orders` | 6 | défaut | Groupes et offsets |
| `audit-short` | 1 | `retention.ms=60000`, `segment.ms=10000` | Rétention en temps |
| `product-prices` | 1 | `cleanup.policy=compact`, segments de 10 s | Compaction |

## 4. Contexte métier

Chez NovaShop, deux services lisent les commandes :

- **Facturation** (groupe `billing`) doit traiter chaque commande une fois, et pouvoir rejouer une période après un incident ;
- **Analytique** (groupe `analytics`) doit monter en charge en ajoutant des instances.

Le service Audit veut un journal court, et le service Catalogue un référentiel de prix qui ne garde que le dernier prix de chaque produit. L'état des groupes s'affiche en fin de notebook.

## 5. Énoncé

1. Faites consommer `orders` par la facturation, puis montrez le lag créé par de nouvelles commandes.
2. Faites travailler deux instances de l'analytique en parallèle, et observez ce qui se passe quand l'une s'arrête.
3. Repositionnez la facturation pour rejouer des commandes, sans jamais appliquer un changement que vous n'avez pas simulé.
4. Prouvez que la rétention supprime les vieux messages d'`audit-short` et que la compaction ne garde que le dernier prix de chaque produit.

## 6. Étapes

### 6.1 Consommer avec un groupe

Un consommateur qui appartient à un groupe valide (« commit ») ses offsets dans le topic interne `__consumer_offsets`. À son redémarrage, il reprend à ces offsets. `auto.offset.reset` ne sert que pour une partition où le groupe n'a encore rien validé : `earliest` lit alors depuis le début, `latest` (valeur par défaut) seulement les nouveaux messages.

**Exemple en ligne de commande** : les groupes existants.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list

**Code fourni** : trois fonctions que la suite du TP réutilise. Lisez leurs commentaires, sans rien modifier. `consume()` crée un consommateur avec les deux réglages de l'étape, `group.id` et `auto.offset.reset` ; `lag()` fait le calcul de la colonne `LAG` de `kafka-consumer-groups.sh --describe`.

In [ ]:
# Code fourni : lire un topic avec un groupe, lire ses offsets validés, calculer son lag
def consume(group, topic, offset_reset):
    """Lit le topic avec un consommateur du groupe, jusqu'à 5 s sans nouveau message ; renvoie les messages lus."""
    consumer = Consumer({
        "bootstrap.servers": BS,
        "group.id": group,                  # le groupe : ses membres se partagent les partitions
        "auto.offset.reset": offset_reset,  # point de départ d'une partition où le groupe n'a rien validé
        "log_level": 4,
    })
    consumer.subscribe([topic])             # le coordinateur du groupe attribue les partitions
    messages, last = [], time.time()
    # Le premier message n'arrive qu'après l'entrée dans le groupe : quelques secondes
    while time.time() - last < 5:
        batch = consumer.consume(100, timeout=1)   # jusqu'à 100 messages ; liste vide si rien en 1 s
        if batch:
            messages += [m for m in batch if m.error() is None]
            last = time.time()
    # Les offsets lus sont validés automatiquement (enable.auto.commit vaut true par défaut),
    # une dernière fois par close(), qui fait aussi quitter le groupe proprement
    consumer.close()
    return messages


def committed_offsets(group, topic):
    """{partition: offset validé par le groupe} ; 0 pour une partition où il n'a rien validé."""
    admin = AdminClient(CONF)
    # Offsets validés du groupe : la colonne CURRENT-OFFSET de kafka-consumer-groups.sh --describe
    result = admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(group)])[group].result()
    committed = {tp.partition: tp.offset for tp in result.topic_partitions if tp.topic == topic}
    # Partition absente de la réponse, ou offset négatif (rien de validé) : on compte 0
    return {p: max(committed.get(p, 0), 0) for p in end_offsets(topic)}


def lag(group, topic):
    """{partition: lag} : le nombre de messages qui restent à lire, offset de fin moins offset validé."""
    committed = committed_offsets(group, topic)
    return {p: end - committed[p] for p, end in end_offsets(topic).items()}

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-console-consumer.sh --bootstrap-server $BS --topic orders --group billing \
  --from-beginning --timeout-ms 10000 | wc -l
kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing
```

In [ ]:
# TODO 6.1 : remplacez les deux ... puis exécutez (une dizaine de secondes)
GROUP = ...         # nom du groupe de la facturation (point 4, contexte métier)
OFFSET_RESET = ...  # "earliest" ou "latest" ? Le groupe n'a encore rien validé, et la facturation
                    # doit traiter les 60 commandes déjà présentes dans orders.

# Code fourni, à lire : rien à modifier en dessous
consumed = []
if filled(GROUP, OFFSET_RESET):
    # Lecture de orders par un consommateur du groupe, avec le point de départ choisi
    consumed = consume(GROUP, "orders", OFFSET_RESET)
    print(len(consumed), "messages lus")
    # Lag après la lecture, partition par partition
    print("lag :", lag(GROUP, "orders"))

In [ ]:
# Vérification 6.1
admin = AdminClient(CONF)
assert len(consumed) == 60, "À revoir : 60 messages lus (%d)" % len(consumed)
print("OK  60 messages lus (%d)" % len(consumed))
assert lag("billing", "orders") == {p: 0 for p in range(6)}, "À revoir : lag nul sur les 6 partitions"
print("OK  lag nul sur les 6 partitions")
state = admin.describe_consumer_groups(["billing"])["billing"].result().state
assert state == ConsumerGroupState.EMPTY, "À revoir : groupe billing sans membre actif : %s" % state.name
print("OK  groupe billing sans membre actif : %s" % state.name)

**Résultat attendu** : 60 messages lus, en une dizaine de secondes. Le lag est nul partout et le groupe est `EMPTY` : il n'a plus de membre, mais ses offsets sont conservés.

### 6.2 Créer et résorber du lag

Le lag d'une partition est l'écart entre son offset de fin et l'offset validé du groupe : le nombre de messages qui restent à traiter.

Équivalent en ligne de commande de l'envoi (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
for i in $(seq 1 10); do
  echo "C008|{\"order_id\":\"O-30$i\",\"customer\":\"C008\",\"amount\":10.0,\"city\":\"Lyon\"}"
done | kafka-console-producer.sh --bootstrap-server $BS --topic orders \
  --reader-property parse.key=true --reader-property 'key.separator=|'
```

**Exécutez la cellule `TODO` une seule fois** : chaque exécution envoie 10 commandes.

In [ ]:
# TODO 6.2 (a) : remplacez les deux ... puis exécutez UNE SEULE FOIS (chaque exécution envoie 10 commandes)
KEY = ...                   # clé des 10 commandes du nouveau client C008. Les commandes d'un client doivent
                            # être traitées dans l'ordre, donc rangées dans la même partition.
PREDICTED_PARTITIONS = ...  # sur combien de partitions le lag de billing va-t-il apparaître ? (un nombre)

# Code fourni, à lire : rien à modifier en dessous
lag_after = {}
if filled(KEY, PREDICTED_PARTITIONS):
    # Les 10 commandes, de O-301 à O-3010, au format JSON du TP2 (%d est remplacé par i)
    values = ['{"order_id":"O-30%d","customer":"C008","amount":10.0,"city":"Lyon"}' % i for i in range(1, 11)]
    send("orders", KEY, values)
    # billing n'a pas encore lu ces messages : ils s'ajoutent à son lag
    lag_after = lag("billing", "orders")

print("lag :", lag_after)

In [ ]:
# Vérification 6.2 (a)
assert lag_after == {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 10}, "À revoir : lag de 10 sur la partition 5 seulement : %s" % lag_after
print("OK  lag de 10 sur la partition 5 seulement : %s" % lag_after)
behind = [p for p, v in lag_after.items() if v > 0]
assert PREDICTED_PARTITIONS == len(behind), "À revoir : prévision %r, lag sur %d partition : une clé, une partition (TP2)" % (PREDICTED_PARTITIONS, len(behind))
print("OK  prévision juste : lag sur %d partition" % len(behind))

**Exemple en ligne de commande** : la même mesure avec l'outil Java.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing

In [ ]:
# TODO 6.2 (b) : remplacez le ... par votre prévision, puis exécutez
PREDICTED_REREAD = ...  # la facturation (groupe billing) est relancée, toujours avec "earliest" :
                        # combien de messages va-t-elle lire, 10 ou 70 ?

# Code fourni, à lire : rien à modifier en dessous
reread = []
if filled(PREDICTED_REREAD):
    # Même groupe et même réglage qu'en 6.1
    reread = consume("billing", "orders", "earliest")

# Nombre de messages relus, et leurs clés sans doublon
print(len(reread), "messages relus :", sorted({m.key().decode() for m in reread}))

In [ ]:
# Vérification 6.2 (b)
assert len(reread) == 10 and {m.key().decode() for m in reread} == {"C008"}, "À revoir : 10 commandes de C008 relues"
print("OK  10 commandes de C008 relues")
assert sum(lag("billing", "orders").values()) == 0, "À revoir : lag résorbé"
print("OK  lag résorbé")
assert PREDICTED_REREAD == len(reread), "À revoir : prévision %r, %d messages relus : le groupe reprend à ses offsets validés" % (PREDICTED_REREAD, len(reread))
print("OK  prévision juste : %d messages relus" % len(reread))

**Résultat attendu** : toutes les commandes de `C008` vont dans la partition 5 ; le lag vaut 10 sur cette partition, 0 ailleurs. La relance lit exactement les 10 nouvelles commandes : le groupe reprend à ses offsets validés.

### 6.3 Deux consommateurs dans un groupe

Les consommateurs d'un même groupe se partagent les partitions : chaque partition est lue par un seul membre. La cellule suivante définit `Instance`, une instance d'un service consommateur qui tourne en arrière-plan dans le notebook, comme un terminal de la version Markdown :

- `Instance("analytics")` démarre un consommateur du groupe `analytics`, abonné à `orders` ;
- `instance.partitions` donne ses partitions, mises à jour en continu ;
- `instance.stop()` le ferme proprement : il quitte le groupe ;
- `Instance.stop_all()` arrête toutes les instances lancées.

In [ ]:
# Code fourni : une instance de service consommateur qui tourne en arrière-plan
class Instance:
    """Une instance d'un service consommateur : un Consumer qui tourne dans un fil d'exécution à part."""

    launched = []  # toutes les instances lancées, pour stop_all()

    def __init__(self, group, topic="orders", **config):
        self.group = group
        self.partitions = []                  # partitions attribuées, mises à jour en continu
        self._stop_event = threading.Event()  # signal d'arrêt, donné par stop()
        # **config ajoute des réglages au consommateur (group.protocol à l'étape 6.5)
        self.consumer = Consumer({"bootstrap.servers": BS, "group.id": group,
                                  "auto.offset.reset": "earliest", "log_level": 4, **config})
        self.consumer.subscribe([topic])
        # La boucle _loop() tourne à part : le notebook reste disponible pendant ce temps
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        Instance.launched.append(self)

    def _loop(self):
        while not self._stop_event.is_set():
            # consume() fait aussi vivre l'appartenance au groupe (rééquilibrages, validations)
            self.consumer.consume(100, timeout=0.5)
            # assignment() : les partitions que le coordinateur a attribuées à ce membre
            self.partitions = sorted(tp.partition for tp in self.consumer.assignment())
        # Départ propre : le coordinateur est prévenu et rééquilibre aussitôt
        self.consumer.close()

    def stop(self):
        """Arrête la boucle, ce qui fait quitter le groupe."""
        self._stop_event.set()
        self._thread.join(15)  # attend la fin de la boucle, 15 s au plus

    @classmethod
    def stop_all(cls):
        """Arrête toutes les instances lancées."""
        for instance in cls.launched:
            instance.stop()
        cls.launched = []

    def __repr__(self):
        # Texte affiché par print(instance)
        return "Instance(%s, partitions %s)" % (self.group, self.partitions)

In [ ]:
# TODO 6.3 : remplacez les deux ... par vos prévisions, puis exécutez (une dizaine de secondes)
PREDICTED_PARTITIONS_EACH = ...  # deux instances du groupe analytics se partagent les 6 partitions
                                 # d'orders : combien chacune en recevra-t-elle ? (un nombre)
PREDICTED_RESET_OK = ...         # pendant que le groupe est actif, on demande de remettre ses offsets
                                 # à 0 : la demande sera-t-elle acceptée ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
a = b = analytics_group = None
reset_errors = []
if filled(PREDICTED_PARTITIONS_EACH, PREDICTED_RESET_OK):
    admin = AdminClient(CONF)
    # Deux instances du même groupe : elles se partagent les partitions d'orders
    a = Instance("analytics")
    b = Instance("analytics")
    # Attente : chacune a reçu ses partitions (entrée dans le groupe, quelques secondes)
    wait_for(lambda: a.partitions and b.partitions, timeout=60, message="affectation des deux instances")

    # Description du groupe : état, protocole, stratégie de répartition, membres
    analytics_group = admin.describe_consumer_groups(["analytics"])["analytics"].result()
    print(analytics_group.state.name, analytics_group.type.name, analytics_group.partition_assignor)
    # Un membre par ligne : identifiant, client, partitions attribuées
    table([(m.member_id, m.client_id, sorted(tp.partition for tp in m.assignment.topic_partitions))
           for m in analytics_group.members], ["membre", "client", "partitions"])

    # Demande de retour à l'offset 0 sur les 6 partitions, groupe actif
    request = ConsumerGroupTopicPartitions("analytics", [TopicPartition("orders", p, 0) for p in range(6)])
    result = admin.alter_consumer_group_offsets([request])["analytics"].result()
    # Pas d'exception en cas de refus : l'erreur est rangée dans chaque partition (tp.error)
    reset_errors = [tp.error.name() for tp in result.topic_partitions if tp.error is not None]

print(a, b)
print("erreurs du repositionnement :", reset_errors)

In [ ]:
# Vérification 6.3
assert (a is not None and b is not None and sorted(a.partitions + b.partitions) == list(range(6))
         and len(a.partitions) == 3), "À revoir : 3 partitions par instance : %s et %s" % (a, b)
print("OK  3 partitions par instance : %s et %s" % (a, b))
assert (analytics_group is not None and analytics_group.state == ConsumerGroupState.STABLE
         and len(analytics_group.members) == 2), "À revoir : groupe STABLE avec 2 membres"
print("OK  groupe STABLE avec 2 membres")
assert analytics_group.partition_assignor == "range", "À revoir : stratégie range"
print("OK  stratégie range")
assert len(reset_errors) == 6 and set(reset_errors) == {"UNKNOWN_MEMBER_ID"}, "À revoir : repositionnement refusé sur les 6 partitions : %s" % sorted(set(reset_errors))
print("OK  repositionnement refusé sur les 6 partitions : %s" % sorted(set(reset_errors)))
assert PREDICTED_PARTITIONS_EACH == len(a.partitions), "À revoir : prévision %r, chaque instance a %d partitions" % (PREDICTED_PARTITIONS_EACH, len(a.partitions))
print("OK  prévision juste : %d partitions par instance" % len(a.partitions))
assert PREDICTED_RESET_OK is False, "À revoir : prévision %r, le repositionnement d'un groupe actif est refusé" % (PREDICTED_RESET_OK,)
print("OK  prévision juste : repositionnement refusé")

**Vue en ligne de commande**, pendant que les deux instances tournent. Le repositionnement est demandé en simulation (`--dry-run`) : l'outil le refuse aussi.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --members --verbose
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --state
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group analytics --topic orders --reset-offsets --to-earliest --dry-run

**Résultat attendu** : chaque instance reçoit 3 partitions, `[0, 1, 2]` et `[3, 4, 5]`. État `STABLE`, 2 membres, stratégie `range`, client `rdkafka`. Le repositionnement est refusé partition par partition (`UNKNOWN_MEMBER_ID`) ; l'outil Java refuse avant d'envoyer quoi que ce soit : `Assignments can only be reset if the group 'analytics' is inactive, but the current state is Stable.`

### 6.4 Départ d'un consommateur

Quand un membre quitte le groupe, le coordinateur rééquilibre : il redistribue ses partitions aux membres restants.

In [ ]:
# TODO 6.4 : remplacez les deux ... par vos prévisions, puis exécutez (quelques secondes)
PREDICTED_PARTITIONS_A = ...  # après l'arrêt de b, combien de partitions pour a ? (un nombre)
PREDICTED_FINAL_STATE = ...   # après l'arrêt de a à son tour, état du groupe analytics :
                              # "STABLE", "EMPTY" ou "DEAD" ?

# Code fourni, à lire : rien à modifier en dessous
states = []  # relevés : (secondes depuis l'arrêt de b, état du groupe)
final_state = None
if filled(PREDICTED_PARTITIONS_A, PREDICTED_FINAL_STATE):
    admin = AdminClient(CONF)
    start = time.time()
    b.stop()  # départ propre : b prévient le coordinateur, qui rééquilibre aussitôt
    # Relevé de l'état du groupe toutes les 0,5 s, pendant 20 s au plus
    while time.time() - start < 20:
        group = admin.describe_consumer_groups(["analytics"])["analytics"].result()
        states.append((round(time.time() - start, 1), group.state.name))
        # Fin du rééquilibrage : groupe STABLE et a possède les 6 partitions
        if group.state == ConsumerGroupState.STABLE and len(a.partitions) == 6:
            break
        time.sleep(0.5)

    a.stop()  # le dernier membre part à son tour
    final_state = admin.describe_consumer_groups(["analytics"])["analytics"].result().state.name

# Les relevés : (secondes depuis l'arrêt de b, état du groupe)
print(states)
print("a :", a, "- état final :", final_state)

In [ ]:
# Vérification 6.4
assert states and states[-1][1] == "STABLE", "À revoir : groupe de nouveau STABLE après le départ de b"
print("OK  groupe de nouveau STABLE après le départ de b")
assert a is not None and a.partitions == list(range(6)), "À revoir : a a reçu les 6 partitions"
print("OK  a a reçu les 6 partitions")
assert final_state == "EMPTY", "À revoir : groupe EMPTY après l'arrêt de a"
print("OK  groupe EMPTY après l'arrêt de a")
assert PREDICTED_PARTITIONS_A == len(a.partitions), "À revoir : prévision %r, a a reçu %d partitions" % (PREDICTED_PARTITIONS_A, len(a.partitions))
print("OK  prévision juste : %d partitions pour a" % len(a.partitions))
assert PREDICTED_FINAL_STATE == final_state, "À revoir : prévision %r, état observé %s" % (PREDICTED_FINAL_STATE, final_state)
print("OK  prévision juste : état %s" % final_state)

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics --state

**Résultat attendu** : en général `PREPARING_REBALANCING` pendant une à deux secondes, puis `STABLE` ; l'instance `a` reçoit les 6 partitions. Après son arrêt, le groupe est `EMPTY`, sans membre.

### 6.5 Le nouveau protocole de groupe

Avec le nouveau protocole (KIP-848, `group.protocol=consumer`), le coordinateur calcule lui-même l'affectation cible, et chaque membre converge vers elle sans arrêt global du groupe. Le protocole classique reste la valeur par défaut des clients en Kafka 4.3.

In [ ]:
# TODO 6.5 (a) : remplacez le ... puis exécutez (une dizaine de secondes)
GROUP_PROTOCOL = ...  # valeur de group.protocol qui active le nouveau protocole : "classic" ou "consumer" ?

# Code fourni, à lire : rien à modifier en dessous
c = d = group_v2 = None
types = {}
if filled(GROUP_PROTOCOL):
    admin = AdminClient(CONF)
    # Deux instances du groupe analytics-v2, avec le protocole choisi
    c = Instance("analytics-v2", **{"group.protocol": GROUP_PROTOCOL})
    d = Instance("analytics-v2", **{"group.protocol": GROUP_PROTOCOL})
    # Attente : les 6 partitions sont réparties entre les deux instances
    wait_for(lambda: c.partitions and d.partitions and len(c.partitions) + len(d.partitions) == 6,
             timeout=60, message="affectation des deux instances")

    # Description du groupe ; en nouveau protocole, chaque membre a aussi une affectation cible
    group_v2 = admin.describe_consumer_groups(["analytics-v2"])["analytics-v2"].result()
    print(group_v2.state.name, group_v2.type.name, group_v2.partition_assignor)
    table([(m.member_id, sorted(tp.partition for tp in m.assignment.topic_partitions),
            sorted(tp.partition for tp in m.target_assignment.topic_partitions)) for m in group_v2.members],
          ["membre", "affectation", "cible"])

    # Tous les groupes du cluster, avec leur protocole : CLASSIC ou CONSUMER
    types = {g.group_id: g.type.name for g in admin.list_consumer_groups().result().valid}

print(c, d)
print(types)

In [ ]:
# Vérification 6.5 (a)
assert group_v2 is not None and group_v2.type == ConsumerGroupType.CONSUMER, "À revoir : analytics-v2 de type CONSUMER"
print("OK  analytics-v2 de type CONSUMER")
assert group_v2.partition_assignor == "uniform", "À revoir : affectation calculée par le serveur : uniform"
print("OK  affectation calculée par le serveur : uniform")
assert len(group_v2.members) == 2, "À revoir : 2 membres"
print("OK  2 membres")
assert (types.get("analytics") == "CLASSIC" and types.get("billing") == "CLASSIC"
         and types.get("analytics-v2") == "CONSUMER"), "À revoir : types des groupes : %s" % types
print("OK  types des groupes : %s" % types)

**Vue en ligne de commande**, pendant que les deux instances tournent :

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group analytics-v2 --members --verbose
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list --type

In [ ]:
# Code fourni : arrêt des deux instances, puis état du groupe analytics-v2
Instance.stop_all()  # arrête les instances encore lancées : ici c et d
state_v2 = (None, None)
if group_v2 is not None:  # seulement si l'étape 6.5 (a) a été faite
    admin = AdminClient(CONF)
    group = admin.describe_consumer_groups(["analytics-v2"])["analytics-v2"].result()
    # État du groupe sans membre, et stratégie d'affectation qu'il garde
    state_v2 = (group.state.name, group.partition_assignor)

print(state_v2)

In [ ]:
# Vérification 6.5 (b)
assert state_v2 == ("EMPTY", "uniform"), "À revoir : analytics-v2 vide, assignateur uniform : %s" % (state_v2,)
print("OK  analytics-v2 vide, assignateur uniform : %s" % (state_v2,))

**Résultat attendu** : `type` `CONSUMER`, assignateur `uniform`, 2 membres dont l'affectation est égale à la cible. Les colonnes `CURRENT-EPOCH` et `TARGET-EPOCH` de l'outil Java sont renseignées. `--list --type` affiche `Consumer` pour `analytics-v2` et `Classic` pour les autres groupes. Après l'arrêt des deux instances, le groupe est `EMPTY`.

### 6.6 Réinitialiser les offsets

Le groupe `billing` est inactif. On peut repositionner ses offsets, en simulant toujours avant d'appliquer.

**Exemple en ligne de commande** : simulation d'un retour au début. `--dry-run` affiche les nouveaux offsets sans rien modifier.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-earliest --dry-run

Équivalent en ligne de commande des `TODO` 6.6 (**ne pas lancer en plus des TODO**) :

```bash
# [tools]
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-earliest --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --to-latest --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --shift-by -5 --execute
kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders \
  --reset-offsets --to-current --export --dry-run > /tmp/billing-offsets.csv
```

**Code fourni** : `reset_offsets()` joue le rôle de `--dry-run` et de `--execute`. Elle reçoit les nouveaux offsets sous forme de dictionnaire `{partition: nouvel offset}`. Sans `apply=True`, elle affiche seulement le tableau des changements prévus.

In [ ]:
# Code fourni : repositionner les offsets d'un groupe, en simulation ou pour de bon
def reset_offsets(group, topic, new_offsets, apply=False):
    """Repositionne le groupe sur new_offsets, un dictionnaire {partition: nouvel offset}.

    apply=False (par défaut) : simulation, comme --dry-run ; affiche le tableau, ne modifie rien.
    apply=True : application, comme --execute ; lève RuntimeError si une partition est refusée.
    """
    committed = committed_offsets(group, topic)
    # Tableau partition / offset validé / nouvel offset : à relire avant d'appliquer
    table([(p, committed.get(p), o) for p, o in sorted(new_offsets.items())],
          ["partition", "offset validé", "nouvel offset"])
    if not apply:
        print("Simulation : rien n'est modifié")
        return
    admin = AdminClient(CONF)
    # Une demande pour le groupe : la liste des (topic, partition, nouvel offset)
    request = ConsumerGroupTopicPartitions(group, [TopicPartition(topic, p, o) for p, o in new_offsets.items()])
    result = admin.alter_consumer_group_offsets([request])[group].result()
    # Un refus ne lève pas d'exception : l'erreur est rangée dans chaque partition (étape 6.3)
    refused = [(tp.partition, tp.error.name()) for tp in result.topic_partitions if tp.error is not None]
    if refused:
        raise RuntimeError("partitions refusées : %s" % refused)
    print("Appliqué")

In [ ]:
# TODO 6.6 (a) : retour au début, en simulation. Remplacez le ... puis exécutez
START_SPEC = ...  # quels offsets pour tout relire : OffsetSpec.earliest() ou OffsetSpec.latest() ?

# Code fourni, à lire : rien à modifier en dessous
starts = {}
if filled(START_SPEC):
    # Offset de chaque partition selon START_SPEC : {partition: offset}
    starts = topic_offsets("orders", START_SPEC)
    # Simulation seule (apply n'est pas donné) : lisez le tableau
    reset_offsets("billing", "orders", starts)

Lisez le tableau : la colonne « nouvel offset » donne-t-elle bien le début de chaque partition ? Si oui, appliquez.

In [ ]:
# TODO 6.6 (b) : application. Remplacez le ... par votre prévision, puis exécutez
PREDICTED_LAG_START = ...  # lag total de billing après le retour au début ? (un nombre ;
                           # orders contient 70 messages)

# Code fourni, à lire : rien à modifier en dessous
lag_start = None
# starts est rempli par l'étape 6.6 (a) : sans simulation, pas d'application
if filled(PREDICTED_LAG_START) and starts:
    # Les offsets de la simulation, cette fois appliqués
    reset_offsets("billing", "orders", starts, apply=True)
    # Lag total : la somme des lags des 6 partitions
    lag_start = sum(lag("billing", "orders").values())

print("lag total après le retour au début :", lag_start)

In [ ]:
# TODO 6.6 (c) : reculer de 5 messages depuis la fin. Remplacez le ... par votre prévision, puis exécutez
PREDICTED_LAG_REWIND = ...  # lag total après un recul de 5 messages sur chaque partition ? (un nombre)
                            # Rappel : la cellule « Outils du TP » a affiché les offsets de fin
                            # d'orders, et deux partitions sont vides.

# Code fourni, à lire : rien à modifier en dessous
lag_rewind = None
if filled(PREDICTED_LAG_REWIND) and starts:
    # 1. Placement à la fin de chaque partition, comme --to-latest
    ends = end_offsets("orders")
    reset_offsets("billing", "orders", ends, apply=True)
    # 2. Nouveaux offsets : 5 avant la fin, sans descendre sous le premier offset disponible.
    #    max() garde la plus grande des deux valeurs : une partition vide reste à son début.
    rewind = {p: max(end - 5, starts[p]) for p, end in ends.items()}
    # 3. Simulation, puis application
    reset_offsets("billing", "orders", rewind)
    reset_offsets("billing", "orders", rewind, apply=True)
    lag_rewind = sum(lag("billing", "orders").values())

print("lag total après le recul :", lag_rewind)

In [ ]:
# Code fourni : export des offsets validés de billing, une ligne "topic,partition,offset" par partition,
# au format de kafka-consumer-groups.sh --export. En production, cet export se fait avant chaque
# repositionnement : il permet de revenir en arrière.
if lag_rewind is not None:  # seulement si l'étape 6.6 (c) a été faite
    with open("/tmp/billing-offsets.csv", "w", newline="") as f:
        writer = csv.writer(f)
        for p, o in sorted(committed_offsets("billing", "orders").items()):
            writer.writerow(["orders", p, o])
    # Contenu du fichier
    print(open("/tmp/billing-offsets.csv").read())

In [ ]:
# Vérification 6.6
assert lag_start == 70, "À revoir : après retour au début, lag égal au nombre de messages : %s" % lag_start
print("OK  après retour au début, lag égal au nombre de messages : %s" % lag_start)
assert lag_rewind == 20, "À revoir : après recul de 5, lag de 20 (5 sur les 4 partitions non vides) : %s" % lag_rewind
print("OK  après recul de 5, lag de 20 (5 sur les 4 partitions non vides) : %s" % lag_rewind)
try:
    lines = open("/tmp/billing-offsets.csv").read().split()
except OSError:
    lines = []
assert len(lines) == 6 and all(l.startswith("orders,") for l in lines), "À revoir : export CSV de 6 lignes"
print("OK  export CSV de 6 lignes")
assert PREDICTED_LAG_START == lag_start, "À revoir : prévision %r, lag observé %s après le retour au début" % (PREDICTED_LAG_START, lag_start)
print("OK  prévision juste : lag de %s après le retour au début" % lag_start)
assert PREDICTED_LAG_REWIND == lag_rewind, "À revoir : prévision %r, lag observé %s : une partition vide ne recule pas" % (PREDICTED_LAG_REWIND, lag_rewind)
print("OK  prévision juste : lag de %s après le recul" % lag_rewind)

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group billing

**Résultat attendu** : après le retour au début, un lag total de 70, tous les messages du topic. Après le recul de 5, un lag de 20 : 5 sur chacune des partitions 0, 1, 2 et 5 ; les partitions vides 3 et 4 restent à 0. Le fichier contient une ligne par partition, de `orders,0,7` à `orders,5,16`.

### 6.7 Défi : retraiter la dernière heure

La facturation a mal traité toutes les commandes reçues depuis une heure. Repositionnez `billing` pour les retraiter, en simulant d'abord.

**Exemple en ligne de commande** : les options de l'outil Java qui parlent de temps.

In [ ]:
!docker exec kafka-tools bash -c "kafka-consumer-groups.sh --help 2>&1 | grep -A2 -E '^--(by-duration|to-datetime)'"

In [ ]:
# TODO 6.7 (défi) : remplacez le ... puis exécutez
#   OffsetSpec.for_timestamp(ms) donne, pour chaque partition, le premier offset dont l'horodatage est
#   postérieur ou égal à ms, en millisecondes depuis le 1er janvier 1970.
#   time.time() donne l'heure actuelle en SECONDES depuis la même date.
ONE_HOUR_AGO_MS = ...  # l'heure qu'il était il y a une heure, en millisecondes (un nombre entier : int(...))

# Code fourni, à lire : rien à modifier en dessous
new_offsets_1h = {}
if filled(ONE_HOUR_AGO_MS):
    # Premier offset de moins d'une heure dans chaque partition, ou -1 s'il n'y en a pas
    since = topic_offsets("orders", OffsetSpec.for_timestamp(ONE_HOUR_AGO_MS))
    ends = end_offsets("orders")
    # Pour chaque partition p : son offset o s'il existe ; sinon (-1), rien à retraiter, on se place à la fin
    new_offsets_1h = {p: (o if o >= 0 else ends[p]) for p, o in since.items()}
    # Simulation, puis application
    reset_offsets("billing", "orders", new_offsets_1h)
    reset_offsets("billing", "orders", new_offsets_1h, apply=True)

print(new_offsets_1h)

In [ ]:
# Vérification 6.7 : comparaison avec la simulation de l'outil Java
output = !docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --group billing --topic orders --reset-offsets --by-duration PT1H --dry-run 2>&1
output = output.n
expected_java = {int(p): int(o) for p, o in re.findall(r"^billing\s+orders\s+(\d+)\s+(\d+)", output, re.M)}
print("outil Java :", expected_java)
assert new_offsets_1h and new_offsets_1h == expected_java, "À revoir : mêmes offsets que --by-duration PT1H"
print("OK  mêmes offsets que --by-duration PT1H")
assert committed_offsets("billing", "orders") == new_offsets_1h, "À revoir : repositionnement appliqué"
print("OK  repositionnement appliqué")

**Résultat attendu** : si le TP2 date de moins d'une heure, l'offset 0 partout, comme `--to-earliest`. Si le déjeuner les sépare, seules les commandes de `C008` (étape 6.2) ont moins d'une heure : la partition 5 revient à l'offset 11, les autres partitions sont placées à la fin.

### 6.8 Rétention en temps

La rétention supprime des **segments** entiers, jamais des messages isolés. Un segment est fermé quand il atteint `segment.bytes` ou `segment.ms` : le message suivant ouvre un nouveau segment. Un segment est supprimé quand son message le plus récent dépasse `retention.ms`, y compris le segment actif (celui où le broker écrit) : le broker en ouvre alors un nouveau, vide.

**Exemple en ligne de commande** : la fréquence de vérification de la rétention, réglage de lab.

In [ ]:
!grep -B1 log.retention.check.interval.ms config/kafka/kafka-1/server.properties

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic audit-short --partitions 1 --replication-factor 3 \
  --config retention.ms=60000 --config segment.ms=10000
seq 1 100 | sed 's/^/audit-/' | kafka-console-producer.sh --bootstrap-server $BS --topic audit-short
sleep 35; echo audit-101 | kafka-console-producer.sh --bootstrap-server $BS --topic audit-short
kafka-get-offsets.sh --bootstrap-server $BS --topic audit-short --time earliest
```

Pourquoi 35 s : plus que `segment.ms` (10 s), pour que `audit-101` ouvre un nouveau segment ; et plus que la période de vérification de la rétention (30 s dans le lab), pour qu'une vérification passe entre l'expiration du premier segment et celle du second.

In [ ]:
# TODO 6.8 : remplacez les trois ... puis exécutez UNE SEULE FOIS (2 à 3 minutes d'attente)
RETENTION_MS = ...            # retention.ms du topic : 1 minute, en millisecondes (un nombre)
SEGMENT_MS = ...              # segment.ms du topic : 10 secondes, en millisecondes (un nombre)
PREDICTED_FIRST_OFFSET = ...  # audit-1 à audit-100 occupent les offsets 0 à 99 du premier segment.
                              # Une fois ce segment supprimé, quel sera le premier offset disponible ?

# Code fourni, à lire : rien à modifier en dessous
retention_duration = None
if filled(RETENTION_MS, SEGMENT_MS, PREDICTED_FIRST_OFFSET):
    admin = AdminClient(CONF)
    # Création du topic : 1 partition, 3 répliques, rétention et durée de segment propres au topic
    # (les valeurs de configuration sont passées en texte, d'où str())
    topic = NewTopic("audit-short", num_partitions=1, replication_factor=3,
                     config={"retention.ms": str(RETENTION_MS), "segment.ms": str(SEGMENT_MS)})
    admin.create_topics([topic])["audit-short"].result()
    # 100 messages sans clé, audit-1 à audit-100 : ils forment le premier segment
    send("audit-short", None, ["audit-%d" % i for i in range(1, 101)])
    print("début :", topic_offsets("audit-short", OffsetSpec.earliest()), end_offsets("audit-short"))

    # 35 s plus tard, audit-101 arrive après segment.ms : il ferme le premier segment et ouvre le suivant
    time.sleep(35)
    send("audit-short", None, ["audit-101"])

    # Attente de la suppression du premier segment : le premier offset disponible atteint 100
    def first_if_deleted():
        first = topic_offsets("audit-short", OffsetSpec.earliest())[0]
        return first if first >= 100 else None

    start = time.time()
    first = wait_for(first_if_deleted, timeout=180, step=5, message="suppression du premier segment")
    retention_duration = round(time.time() - start)
    print("premier offset observé :", first, "- durée de la rétention :", retention_duration, "s")

In [ ]:
# Vérification 6.8
admin = AdminClient(CONF)
configs = list(admin.describe_configs([ConfigResource("topic", "audit-short")]).values())[0].result()
settings = (configs["retention.ms"].value, configs["segment.ms"].value)
assert settings == ("60000", "10000"), "À revoir : retention.ms=60000 et segment.ms=10000 : %s" % (settings,)
print("OK  retention.ms=60000 et segment.ms=10000")
assert retention_duration is not None, "À revoir : suppression du premier segment attendue (%s s)" % retention_duration
print("OK  suppression du premier segment attendue (%s s)" % retention_duration)
first_offset = topic_offsets("audit-short", OffsetSpec.earliest())[0]
assert first_offset in (100, 101), "À revoir : premier offset disponible : %d (100, puis 101 quand le segment actif expire à son tour)" % first_offset
print("OK  premier offset disponible : %d (100, puis 101 quand le segment actif expire à son tour)" % first_offset)
assert end_offsets("audit-short") == {0: 101}, "À revoir : 101 messages écrits"
print("OK  101 messages écrits")
assert PREDICTED_FIRST_OFFSET == 100, "À revoir : prévision %r : le segment supprimé contenait les offsets 0 à 99" % (PREDICTED_FIRST_OFFSET,)
print("OK  prévision juste : premier offset 100 après la suppression du premier segment")

In [ ]:
!docker compose exec -T kafka-1 ls -l /var/lib/kafka/data/audit-short-0

**Résultat attendu** :

- d'abord `{0: 0}` et `{0: 100}` ;
- entre 1 min et 1 min 40 après le 100e message, le premier offset passe à 100. Dans le répertoire de la partition, l'ancien segment est renommé `00000000000000000000.log.deleted`, puis effacé une minute plus tard ; le segment actif est `00000000000000000100.log` ;
- 30 à 60 s plus tard, le premier offset passe à 101 : le segment actif, qui ne contient que `audit-101`, a expiré à son tour. La vérification accepte donc 100 ou 101, selon le moment où vous l'exécutez.

### 6.9 Compaction

Un topic compacté (`cleanup.policy=compact`) garde au moins le dernier message de chaque clé. Le nettoyeur (« log cleaner ») ne travaille que sur les segments fermés.

**Code fourni** : la fonction ci-dessous lit tous les messages encore présents dans une partition. Avec un topic compacté, les offsets ne se suivent plus : elle lit jusqu'au dernier offset du topic au lieu de compter les messages.

In [ ]:
# Code fourni : lire tous les messages encore présents dans une partition
def read_all(topic, partition=0):
    """Messages encore présents dans une partition : liste de (offset, key, value)."""
    end = end_offsets(topic)[partition]
    # Lecture sans groupe actif : assign() choisit la partition, sans validation d'offset
    consumer = Consumer({"bootstrap.servers": BS, "group.id": "tp3-lecture",
                         "enable.auto.commit": False, "log_level": 4})
    consumer.assign([TopicPartition(topic, partition, OFFSET_BEGINNING)])
    messages = []
    # Lecture jusqu'au dernier offset (end - 1), ou jusqu'à 5 s sans message
    while end and (not messages or messages[-1][0] < end - 1):
        batch = consumer.consume(100, timeout=5)
        if not batch:
            break
        for m in batch:
            if m.error() is None:
                # Clé et valeur en texte ; None pour une valeur nulle (tombstone, étape 6.10)
                key = m.key().decode() if m.key() is not None else None
                value = m.value().decode() if m.value() is not None else None
                messages.append((m.offset(), key, value))
    consumer.close()
    return messages

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic product-prices --partitions 1 --replication-factor 3 \
  --config cleanup.policy=compact --config segment.ms=10000 \
  --config min.cleanable.dirty.ratio=0.01 --config delete.retention.ms=1000
kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|' < /opt/lab/data/product-prices.txt
sleep 12
echo 'P-500|9.99' | kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|'
```

In [ ]:
# TODO 6.9 : remplacez les deux ... puis exécutez UNE SEULE FOIS (une minute d'attente environ)
CLEANUP_POLICY = ...   # cleanup.policy : "delete" ou "compact" ? Le Catalogue ne veut garder que
                       # le dernier prix de chaque produit.
PREDICTED_COUNT = ...  # 20 prix pour 4 produits (P-100 à P-400), puis un prix pour P-500 :
                       # combien de messages resteront après le nettoyage ? (un nombre)

# Code fourni, à lire : rien à modifier en dessous
prices = []
if filled(CLEANUP_POLICY, PREDICTED_COUNT):
    admin = AdminClient(CONF)
    # Topic compacté. Réglages de lab pour voir le nettoyage en une minute : segments de 10 s,
    # nettoyage dès 1 % de messages remplacés, tombstones gardés 1 s (étape 6.10)
    topic = NewTopic("product-prices", num_partitions=1, replication_factor=3, config={
        "cleanup.policy": CLEANUP_POLICY, "segment.ms": "10000",
        "min.cleanable.dirty.ratio": "0.01", "delete.retention.ms": "1000"})
    admin.create_topics([topic])["product-prices"].result()

    # Les 20 lignes du fichier, au format "produit|prix" : le produit est la clé, le prix la valeur
    with open("data/product-prices.txt") as f:
        for line in f.read().splitlines():
            product, value = line.split("|", 1)
            send("product-prices", product, [value])

    # P-500 arrive après segment.ms : il ferme le segment des 20 prix, que le nettoyeur peut compacter
    time.sleep(12)
    send("product-prices", "P-500", ["9.99"])
    print(len(read_all("product-prices")), "messages juste après l'envoi de P-500")

    # Attente du nettoyage : il ne doit plus rester que 5 messages
    wait_for(lambda: len(read_all("product-prices")) == 5, timeout=180, step=5, message="compaction")
    prices = read_all("product-prices")

table(prices, ["offset", "clé", "valeur"])

In [ ]:
# Vérification 6.9
assert len(prices) == 5 and sorted(m[1] for m in prices) == ["P-100", "P-200", "P-300", "P-400", "P-500"], "À revoir : un seul prix par produit"
print("OK  un seul prix par produit")
assert prices[0][0] > 0, "À revoir : les offsets conservés ne se suivent pas : premier offset %s" % (prices[0][0] if prices else None)
print("OK  les offsets conservés ne se suivent pas : premier offset %s" % (prices[0][0] if prices else None))
assert PREDICTED_COUNT == len(prices), "À revoir : prévision %r, %d messages restants" % (PREDICTED_COUNT, len(prices))
print("OK  prévision juste : %d messages restants" % len(prices))

**Résultat attendu** : 21 messages juste après l'envoi de `P-500`, puis 5 en moins d'une minute : le dernier prix de `P-100`, `P-200`, `P-300`, `P-400` et `P-500`, aux offsets 16 à 20.

### 6.10 Tombstone

Pour supprimer une clé d'un topic compacté, on envoie un **tombstone** : un message avec cette clé et une valeur nulle. Le nettoyeur supprime d'abord les anciennes valeurs, puis, plus tard, le tombstone lui-même.

Équivalent en ligne de commande de l'envoi du tombstone (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
echo 'P-500|NULL' | kafka-console-producer.sh --bootstrap-server $BS --topic product-prices \
  --reader-property parse.key=true --reader-property 'key.separator=|' --reader-property null.marker=NULL
```

In [ ]:
# TODO 6.10 : remplacez le ... puis exécutez UNE SEULE FOIS (une à trois minutes)
TOMBSTONE_VALUE = ...  # valeur du message qui supprime la clé P-500. En Python, la valeur nulle s'écrit None.

# Code fourni, à lire : rien à modifier en dessous
steps = []
if filled(TOMBSTONE_VALUE):
    def values_p500():
        """Valeurs encore lisibles pour la clé P-500."""
        return [v for (o, k, v) in read_all("product-prices") if k == "P-500"]

    # Envoi du message de suppression, puis premier relevé : (secondes écoulées, valeurs de P-500)
    send("product-prices", "P-500", [TOMBSTONE_VALUE])
    start = time.time()
    steps = [(0, values_p500())]

    # Toutes les 15 s, pendant 3 minutes au plus, tant que P-500 a encore une valeur lisible
    i = 0
    while steps[-1][1] and time.time() - start < 180:
        time.sleep(15)
        i += 1
        # Nouvelle écriture d'un autre produit : elle ferme le segment et relance le nettoyeur
        send("product-prices", "P-90%d" % i, ["0.9%d" % i])
        steps.append((round(time.time() - start), values_p500()))

for t, values in steps:
    print("t+%3d s : %s" % (t, values))

In [ ]:
# Vérification 6.10
assert steps and steps[0][1] == ["9.99", None], "À revoir : au début, l'ancienne valeur et le tombstone coexistent"
print("OK  au début, l'ancienne valeur et le tombstone coexistent")
assert steps[-1][1] == [], "À revoir : à la fin, plus aucune trace de P-500"
print("OK  à la fin, plus aucune trace de P-500")

**Résultat attendu** :

- d'abord, l'ancienne valeur `9.99` et le tombstone `None` coexistent ;
- ensuite, l'ancienne valeur disparaît, le tombstone reste ;
- enfin, environ une minute après l'envoi, plus aucune valeur pour `P-500`.

## État des groupes

La cellule suivante résume l'état des groupes de consommateurs à la fin du TP.

In [ ]:
# Code fourni : une ligne par groupe du TP, avec son protocole, son état, ses membres et son lag
def group_state(name):
    admin = AdminClient(CONF)
    g = admin.describe_consumer_groups([name])[name].result()
    behind = lag(name, "orders")
    return (name, g.type.name, g.state.name, len(g.members), sum(behind.values()) if behind else "?")


admin = AdminClient(CONF)
# Groupes présents sur le cluster ; un groupe du TP absent est signalé comme tel
existing = {g.group_id for g in admin.list_consumer_groups().result().valid}
lines = []
for name in ("billing", "analytics", "analytics-v2"):
    if name not in existing:
        lines.append((name, "absent", "", "", ""))
        continue
    lines.append(group_state(name))
table(lines, ["Groupe", "Type", "État", "Membres", "Lag sur orders"])

**Attendu** : les trois groupes `EMPTY`, sans membre. `billing` est `CLASSIC`, avec le lag laissé par l'étape 6.7 ; `analytics` est `CLASSIC` et `analytics-v2` est `CONSUMER`, sans lag. Ne supprimez pas ces groupes : le TP6 les réutilise.

## 7. Questions de réflexion

1. Pourquoi la relance de la facturation en 6.2 lit-elle 10 messages et non 70, alors que `auto.offset.reset` vaut `earliest` ?
2. Que se passerait-il avec 8 instances dans le groupe `analytics` ?
3. Pourquoi Kafka refuse-t-il de repositionner les offsets d'un groupe actif ? Comment procéder en production ?
4. Qu'apporte le nouveau protocole de groupe (KIP-848) lors d'un rééquilibrage ?
5. Pourquoi le message `audit-101` a-t-il survécu aux 100 premiers, alors que la rétention est la même pour tous ?
6. Un consommateur qui lit `product-prices` depuis le début voit-il toujours exactement un message par produit ?
7. Pourquoi un tombstone n'est-il pas supprimé immédiatement ?